# Component Comparison

Compare chunking strategies — **FixedSize**, **Sentence**, and
**RecursiveMarkdown** — on the same document.  We look at chunk count,
size statistics, and text previews.

In [ ]:
# ── Change these to use your own document and API keys ──
PDF_PATH = "../tests/fixtures/sample.pdf"
ENV_PATH = "../tests/.env"
# ────────────────────────────────────────────────────────

import os, logging
from dotenv import load_dotenv

load_dotenv(ENV_PATH)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
print("Setup complete ✓")

## 1 — Run OCR to get a shared document

In [ ]:
from ragbandit.documents import (
    DocumentPipeline,
    MistralOCR,
    FixedSizeChunker,
    SentenceChunker,
    RecursiveMarkdownChunker,
)

# Run OCR once to get a shared document
pipeline = DocumentPipeline(ocr_processor=MistralOCR(api_key=MISTRAL_API_KEY))
ocr_result = pipeline.run_ocr(PDF_PATH)
print(f"OCR done — {len(ocr_result.pages)} pages")

## 2 — Define chunkers and a helper

In [ ]:
from ragbandit.documents.refiners.base_refiner import BaseRefiner

# Convert OCRResult to RefiningResult so chunkers accept it
ref_result = BaseRefiner.ensure_refining_result(ocr_result)

chunkers = {
    "FixedSize(1000/200)": FixedSizeChunker(chunk_size=1000, overlap=200),
    "Sentence(5/1)": SentenceChunker(sentences_per_chunk=5, sentence_overlap=1),
    "RecursiveMarkdown(1000/100)": RecursiveMarkdownChunker(chunk_size=1000, overlap=100),
}


def print_stats(name, chunk_result):
    chunks = chunk_result.chunks
    sizes = [len(c.text) for c in chunks]
    print(f"\n{'─' * 50}")
    print(f"  {name}")
    print(f"{'─' * 50}")
    print(f"  Chunks : {len(chunks)}")
    print(f"  Min    : {min(sizes)} chars")
    print(f"  Max    : {max(sizes)} chars")
    print(f"  Avg    : {sum(sizes) // len(sizes)} chars")
    for i, c in enumerate(chunks[:2]):
        preview = c.text[:100].replace('\n', ' ')
        print(f"  Chunk {i}: {preview}...")

## 3 — Run all chunkers and compare

In [ ]:
results = {}
for name, chunker in chunkers.items():
    cr = chunker.chunk(ref_result)
    results[name] = cr
    print_stats(name, cr)

## 4 — Side-by-side summary table

In [ ]:
print(f"{'Chunker':<35} {'Chunks':>6} {'Min':>6} {'Max':>6} {'Avg':>6}")
print("─" * 65)
for name, cr in results.items():
    sizes = [len(c.text) for c in cr.chunks]
    print(
        f"{name:<35} {len(cr.chunks):>6} "
        f"{min(sizes):>6} {max(sizes):>6} {sum(sizes)//len(sizes):>6}"
    )